In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size':20})
import os
import pickle

In [3]:
data_dir = '../Data'
# asv_name = 'tblcounts_asv_wide.csv'
# asv_path = os.path.join(data_dir, asv_name)
# asv_df = pd.read_csv(asv_path)

In [4]:
drugs_name = 'tbldrug.csv'
drugs_path = os.path.join(data_dir, drugs_name)
drugs_df = pd.read_csv(drugs_path)

/tmp/tmp.r5UZGVVEJ7/ipykernel_1050984/2872359644.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  drugs_df = pd.read_csv(drugs_path)


In [5]:
samples_name = 'tblASVsamples.csv'
samples_path = os.path.join(data_dir, samples_name)
samples_df = pd.read_csv(samples_path)

In [6]:
# print(asv_df.head())
print(drugs_df.head())
print(samples_df.head())

  PatientID  StartTimepoint  StopTimepoint         Factor  \
0      1000            -160           -160  ciprofloxacin   
1      1000            -160           -160    fluconazole   
2      1000            -151           -151      aztreonam   
3      1000            -151           -151     vancomycin   
4      1000            -150           -150      aztreonam   

                    Category        Route  StartDayRelativeToNearestHCT  \
0                 quinolones  intravenous                          -169   
1                antifungals  intravenous                          -169   
2  miscellaneous antibiotics  intravenous                          -160   
3   glycopeptide antibiotics  intravenous                          -160   
4  miscellaneous antibiotics  intravenous                          -159   

   StopDayRelativeToNearestHCT  
0                         -169  
1                         -169  
2                         -160  
3                         -160  
4                

In [18]:
samples_df['PatientID'] = samples_df['PatientID'].astype(str)
drugs_df['PatientID'] = drugs_df['PatientID'].astype(str)

# 2. Sort chronologically
samples_df = samples_df.sort_values(by=['PatientID', 'DayRelativeToNearestHCT'])
drugs_df = drugs_df.sort_values(by=['PatientID', 'StartDayRelativeToNearestHCT'])

paired_data = []

# 3. Iterate over patients based on sample availability
for patient_id, patient_samples in samples_df.groupby('PatientID'):
    # A patient needs at least 2 consecutive samples to form an X -> Y pair
    if len(patient_samples) < 2:
        continue
        
    # Isolate this patient's drug history
    patient_drugs = drugs_df[drugs_df['PatientID'] == patient_id]
    
    # 4. Iterate through consecutive sample pairs
    for i in range(len(patient_samples) - 1):
        sample_x_row = patient_samples.iloc[i]
        sample_y_row = patient_samples.iloc[i + 1]
        
        sample_x = sample_x_row['SampleID']
        day_x = sample_x_row['DayRelativeToNearestHCT']
        
        sample_y = sample_y_row['SampleID']
        day_y = sample_y_row['DayRelativeToNearestHCT']
        
        # 5a. Find drugs STARTED in this interval
        started_drugs_interval = patient_drugs[
            (patient_drugs['StartDayRelativeToNearestHCT'] > day_x) & 
            (patient_drugs['StartDayRelativeToNearestHCT'] <= day_y)
        ]
        
        if not started_drugs_interval.empty:
            new_drugs = sorted(started_drugs_interval['Factor'].dropna().unique())
            started_combo = ' + '.join(new_drugs)
        else:
            started_combo = 'None'
            
        # 5b. Find drugs STOPPED in this interval
        stopped_drugs_interval = patient_drugs[
            (patient_drugs['StopDayRelativeToNearestHCT'] > day_x) & 
            (patient_drugs['StopDayRelativeToNearestHCT'] <= day_y)
        ]
        
        if not stopped_drugs_interval.empty:
            stopped_drugs = sorted(stopped_drugs_interval['Factor'].dropna().unique())
            stopped_combo = ' + '.join(stopped_drugs)
        else:
            stopped_combo = 'None'
            
        # 6. Append the complete profile for this interval
        paired_data.append({
            'PatientID': patient_id,
            'SampleID_X': sample_x,
            'Day_X': day_x,
            'SampleID_Y': sample_y,
            'Day_Y': day_y,
            'Time_Delta_Days': day_y - day_x,
            'Drugs_Started': started_combo,
            'Drugs_Stopped': stopped_combo
        })

# 7. Create the final DataFrame
paired_df = pd.DataFrame(paired_data)

print(paired_df.head(10))

  PatientID SampleID_X  Day_X SampleID_Y  Day_Y  Time_Delta_Days  \
0      1000      1000A   -9.0      1000B   -4.0              5.0   
1      1000      1000B   -4.0      1000C    6.0             10.0   
2      1000      1000C    6.0      1000D    9.0              3.0   
3      1000      1000D    9.0      1000E   13.0              4.0   
4      1001      1001A   -3.0      1001B    0.0              3.0   
5      1001      1001B    0.0      1001D   13.0             13.0   
6      1001      1001D   13.0      1001E   16.0              3.0   
7      1001      1001E   16.0      1001F   31.0             15.0   
8      1002      1002A  -34.0      1002B   -1.0             33.0   
9      1002      1002B   -1.0      1002C    3.0              4.0   

                                       Drugs_Started  \
0  aztreonam + cefepime + ciprofloxacin + vancomycin   
1                                               None   
2                                       voriconazole   
3                          

In [19]:
len(paired_df)

10862

In [20]:
print(paired_df[paired_df['PatientID']=='986'])

     PatientID SampleID_X  Day_X SampleID_Y  Day_Y  Time_Delta_Days  \
7429       986       986A    3.0       986B   17.0             14.0   

                                          Drugs_Started  \
7429  acyclovir + linezolid + vancomycin + voriconazole   

                                          Drugs_Stopped  
7429  acyclovir + linezolid + micafungin + piperacil...  


In [21]:
print(paired_df['PatientID'].unique())

['1000' '1001' '1002' ... 'pt_with_samples_642_643'
 'pt_with_samples_651_652' 'pt_with_samples_833_883']
